# 0. Libraries and parameters

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from scipy.stats import norm

In [2]:
# PARAMETERS
N_INPUT = 7

---
# 1. Input data
We need a Heston solver to generate synthetic data to feed the NN.

---
# 2. Build the model
Following Brigo et al., we will build a FCNN (fully-connected neural network), with ELU activation, and a small regularization. With this NN we have $$(\Theta, m, \tau)\to \sigma_{imp}.$$

TODO: maybe improve optimizer and loss function.

In [8]:
def build_heston_surrogate():
    inputs = keras.Input(shape=(N_INPUT,), name="heston_params_m_tau")

    x = layers.Dense(256, activation="elu", kernel_regularizer=regularizers.l2(1e-6))(inputs)
    x = layers.Dense(256, activation="elu", kernel_regularizer=regularizers.l2(1e-6))(x)
    x = layers.Dense(256, activation="elu", kernel_regularizer=regularizers.l2(1e-6))(x)
    x = layers.Dense(128, activation="elu", kernel_regularizer=regularizers.l2(1e-6))(x)

    # IV on this (m, tau)
    outputs = layers.Dense(1, activation="linear", name="iv")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="NN_Heston_Surrogate")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="mse")

    return model

In [9]:
model = build_heston_surrogate()
model.summary()

Model: "NN_Heston_Surrogate"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ heston_params_m_tau             │ (None, 7)              │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ iv (Dense)                      │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 166,657 (651.00 KB)

 Trainable params: 166,657 (651.00 KB)

 Non-trainable params: 0 (0.00 B)

---
# 3. Training the model

---
# 4. Evaluations
The idea is compare with generated SABR data.


In [ ]:
accuracy_score(y_test, y_hat)

---
# 5. Save and reload

In [ ]:
model.save('test_model')

In [ ]:
del model # deleting the existing model from memory

In [ ]:
model = load_model('test_model') # loading the saved model